In [3]:
from joblib import load

# Base path for joblib files
base_path = '../../data/processed/joblib_data/'

# List of all data object names
data_object_names = [
    'X_test',
    'X_train',
    'X_train_scaled_df',
    'X_unseen',
    'X_unseen_scaled_df',
    'X_val',
    'X_val_scaled_df',
    'y_train',
    'y_unseen',
    'y_val'
]

# Load each object using joblib and store them in a dictionary
loaded_data_objects = {name: load(f'{base_path}{name}.joblib') for name in data_object_names}

# Access each loaded object
X_test = loaded_data_objects['X_test']
X_train = loaded_data_objects['X_train']
X_train_scaled_df = loaded_data_objects['X_train_scaled_df']
X_unseen = loaded_data_objects['X_unseen']
X_unseen_scaled_df = loaded_data_objects['X_unseen_scaled_df']
X_val = loaded_data_objects['X_val']
X_val_scaled_df = loaded_data_objects['X_val_scaled_df']
y_train = loaded_data_objects['y_train']
y_unseen = loaded_data_objects['y_unseen']
y_val = loaded_data_objects['y_val']


In [7]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc, classification_report, confusion_matrix
import numpy as np

# Calculate scale_pos_weight using the given formula
scale_pos_weight = np.sqrt((y_train == 0).sum() / (y_train == 1).sum())

# Define base models
base_models = [
    ('xgb', xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42)),
    ('lgb', lgb.LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=42))
]

# Define stacking classifier
stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())

# Train stacking model
stack_model.fit(X_train_scaled_df, y_train)

# Model Evaluation
y_pred = stack_model.predict(X_val_scaled_df)
y_pred_prob = stack_model.predict_proba(X_val_scaled_df)[:, 1]

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

# Confusion Matrix
conf_matrix = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = conf_matrix.ravel()

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_val, y_pred))
print("Confusion Matrix:")
print(conf_matrix)
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'True Negatives (TN): {tn}')
print(f'False Negatives (FN): {fn}')


[LightGBM] [Info] Number of positive: 5254, number of negative: 902418
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007198 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1464
[LightGBM] [Info] Number of data points in the train set: 907672, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005788 -> initscore=-5.146088
[LightGBM] [Info] Start training from score -5.146088
[LightGBM] [Info] Number of positive: 4203, number of negative: 721934
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003917 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1464
[LightGBM] [Info] Number of data points in the train set: 726137, number of used features: 12
[LightGBM] [Info

In [8]:
# Model Evaluation on Unseen Data
y_unseen_pred = stack_model.predict(X_unseen_scaled_df)
y_unseen_pred_prob = stack_model.predict_proba(X_unseen_scaled_df)[:, 1]

# Calculate Metrics
accuracy_unseen = accuracy_score(y_unseen, y_unseen_pred)
precision_unseen = precision_score(y_unseen, y_unseen_pred)
recall_unseen = recall_score(y_unseen, y_unseen_pred)
f1_unseen = f1_score(y_unseen, y_unseen_pred)
roc_auc_unseen = roc_auc_score(y_unseen, y_unseen_pred_prob)
precision_vals_unseen, recall_vals_unseen, _ = precision_recall_curve(y_unseen, y_unseen_pred_prob)
pr_auc_unseen = auc(recall_vals_unseen, precision_vals_unseen)

# Confusion Matrix for Unseen Data
conf_matrix_unseen = confusion_matrix(y_unseen, y_unseen_pred)
tn_unseen, fp_unseen, fn_unseen, tp_unseen = conf_matrix_unseen.ravel()

print(f'Accuracy (Unseen): {accuracy_unseen:.4f}')
print(f'Precision (Unseen): {precision_unseen:.4f}')
print(f'Recall (Unseen): {recall_unseen:.4f}')
print(f'F1 Score (Unseen): {f1_unseen:.4f}')
print(f'ROC-AUC (Unseen): {roc_auc_unseen:.4f}')
print(f'PR AUC (Unseen): {pr_auc_unseen:.4f}')
print("Classification Report (Unseen):")
print(classification_report(y_unseen, y_unseen_pred))
print("Confusion Matrix (Unseen):")
print(conf_matrix_unseen)
print(f'True Positives (TP) (Unseen): {tp_unseen}')
print(f'False Positives (FP) (Unseen): {fp_unseen}')
print(f'True Negatives (TN) (Unseen): {tn_unseen}')
print(f'False Negatives (FN) (Unseen): {fn_unseen}')


Accuracy (Unseen): 0.9976
Precision (Unseen): 0.7783
Recall (Unseen): 0.5156
F1 Score (Unseen): 0.6203
ROC-AUC (Unseen): 0.9752
PR AUC (Unseen): 0.6101
Classification Report (Unseen):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.78      0.52      0.62      2145

    accuracy                           1.00    555719
   macro avg       0.89      0.76      0.81    555719
weighted avg       1.00      1.00      1.00    555719

Confusion Matrix (Unseen):
[[553259    315]
 [  1039   1106]]
True Positives (TP) (Unseen): 1106
False Positives (FP) (Unseen): 315
True Negatives (TN) (Unseen): 553259
False Negatives (FN) (Unseen): 1039


In [10]:
%%time
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import joblib

# Initialize StratifiedKFold
strat_k_fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define the parameter grid for XGBoost
xgb_param_grid = {
    'scale_pos_weight': Integer(1, 1000),
    'n_estimators': Integer(50, 200),
    'max_depth': Integer(1, 30),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform')
}

# Initialize XGBoost model
xgb_model = xgb.XGBClassifier(random_state=42)

# Initialize the Bayesian Search CV for XGBoost
xgb_bayes_search = BayesSearchCV(
    estimator=xgb_model,
    search_spaces=xgb_param_grid,
    cv=strat_k_fold,
    scoring='roc_auc',
    n_jobs=-1,
    n_iter=32,  # Number of parameter settings that are sampled
    random_state=42,
    refit=True
)

# Perform the search for XGBoost
with joblib.parallel_backend('threading'):
    xgb_bayes_search.fit(X_train_scaled_df, y_train)

# Best parameters and score for XGBoost
print("Best Parameters for XGBoost:", xgb_bayes_search.best_params_)
print("Best Cross-Validation ROC AUC Score for XGBoost:", xgb_bayes_search.best_score_)

# Define the parameter grid for LightGBM
lgb_param_grid = {
    'scale_pos_weight': Integer(1, 1000),
    'n_estimators': Integer(50, 200),
    'max_depth': Integer(1, 30),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform')
}

# Initialize LightGBM model
lgb_model = lgb.LGBMClassifier(random_state=42)

# Initialize the Bayesian Search CV for LightGBM
lgb_bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=lgb_param_grid,
    cv=strat_k_fold,
    scoring='roc_auc',
    n_jobs=-1,
    n_iter=32,  # Number of parameter settings that are sampled
    random_state=42,
    refit=True
)

# Perform the search for LightGBM
with joblib.parallel_backend('threading'):
    lgb_bayes_search.fit(X_train_scaled_df, y_train)

# Best parameters and score for LightGBM
print("Best Parameters for LightGBM:", lgb_bayes_search.best_params_)
print("Best Cross-Validation ROC AUC Score for LightGBM:", lgb_bayes_search.best_score_)


Best Parameters for XGBoost: OrderedDict([('learning_rate', 0.12124056477830636), ('max_depth', 29), ('n_estimators', 200), ('scale_pos_weight', 1)])
Best Cross-Validation ROC AUC Score for XGBoost: 0.9983292674793024
[LightGBM] [Info] Number of positive: 5254, number of negative: 902418
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007422 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1464
[LightGBM] [Info] Number of data points in the train set: 907672, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005788 -> initscore=-5.146088
[LightGBM] [Info] Start training from score -5.146088
Best Parameters for LightGBM: OrderedDict([('learning_rate', 0.029123317607560525), ('max_depth', 30), ('n_estimators', 200), ('scale_pos_weight', 1)])
Best Cross-Validation ROC AUC Score for LightGBM: 0.9971517359371859
CP

In [11]:
%%time
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc, classification_report, confusion_matrix

# Initialize the tuned base models
tuned_xgb_model = xgb.XGBClassifier(**xgb_bayes_search.best_params_)
tuned_lgb_model = lgb.LGBMClassifier(**lgb_bayes_search.best_params_)

# Define base models
base_models = [
    ('xgb', tuned_xgb_model),
    ('lgb', tuned_lgb_model)
]

# Define stacking classifier
stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_scaled_df, y_train)

# Train stacking model with resampled data
stack_model.fit(X_resampled, y_resampled)

[LightGBM] [Info] Number of positive: 902418, number of negative: 902418
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006797 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2804
[LightGBM] [Info] Number of data points in the train set: 1804836, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 721934, number of negative: 721934
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005343 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2804
[LightGBM] [Info] Number of data points in the train set: 1443868, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> inits

StackingClassifier(estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_...
                                              max_delta_step=None, max_depth=29,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=200, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=None, ...)),
                               ('lgb',
                                LGBMClassifier(learning_rate=0.029123317607560525,
                                               max_depth=30, n_estimators=200,
                                               scale_pos_weight=1))],
                   final_estimator=LogisticRegression())

In [12]:
# Model Evaluation on Validation Data
y_pred = stack_model.predict(X_val_scaled_df)
y_pred_prob = stack_model.predict_proba(X_val_scaled_df)[:, 1]

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

conf_matrix = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = conf_matrix.ravel()

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_val, y_pred))
print("Confusion Matrix:")
print(conf_matrix)
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'True Negatives (TN): {tn}')
print(f'False Negatives (FN): {fn}')

Accuracy: 0.9983
Precision: 0.8743
Recall: 0.8184
F1 Score: 0.8454
ROC-AUC: 0.9959
PR AUC: 0.8922
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    386751
           1       0.87      0.82      0.85      2252

    accuracy                           1.00    389003
   macro avg       0.94      0.91      0.92    389003
weighted avg       1.00      1.00      1.00    389003

Confusion Matrix:
[[386486    265]
 [   409   1843]]
True Positives (TP): 1843
False Positives (FP): 265
True Negatives (TN): 386486
False Negatives (FN): 409


In [13]:
# Model Evaluation on Unseen Data
y_unseen_pred = stack_model.predict(X_unseen_scaled_df)
y_unseen_pred_prob = stack_model.predict_proba(X_unseen_scaled_df)[:, 1]

accuracy_unseen = accuracy_score(y_unseen, y_unseen_pred)
precision_unseen = precision_score(y_unseen, y_unseen_pred)
recall_unseen = recall_score(y_unseen, y_unseen_pred)
f1_unseen = f1_score(y_unseen, y_unseen_pred)
roc_auc_unseen = roc_auc_score(y_unseen, y_unseen_pred_prob)
precision_vals_unseen, recall_vals_unseen, _ = precision_recall_curve(y_unseen, y_unseen_pred_prob)
pr_auc_unseen = auc(recall_vals_unseen, precision_vals_unseen)

conf_matrix_unseen = confusion_matrix(y_unseen, y_unseen_pred)
tn_unseen, fp_unseen, fn_unseen, tp_unseen = conf_matrix_unseen.ravel()

print(f'Accuracy (Unseen): {accuracy_unseen:.4f}')
print(f'Precision (Unseen): {precision_unseen:.4f}')
print(f'Recall (Unseen): {recall_unseen:.4f}')
print(f'F1 Score (Unseen): {f1_unseen:.4f}')
print(f'ROC-AUC (Unseen): {roc_auc_unseen:.4f}')
print(f'PR AUC (Unseen): {pr_auc_unseen:.4f}')
print("Classification Report (Unseen):")
print(classification_report(y_unseen, y_unseen_pred))
print("Confusion Matrix (Unseen):")
print(conf_matrix_unseen)
print(f'True Positives (TP) (Unseen): {tp_unseen}')
print(f'False Positives (FP) (Unseen): {fp_unseen}')
print(f'True Negatives (TN) (Unseen): {tn_unseen}')
print(f'False Negatives (FN) (Unseen): {fn_unseen}')

Accuracy (Unseen): 0.9975
Precision (Unseen): 0.7469
Recall (Unseen): 0.5296
F1 Score (Unseen): 0.6197
ROC-AUC (Unseen): 0.9714
PR AUC (Unseen): 0.5741
Classification Report (Unseen):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.75      0.53      0.62      2145

    accuracy                           1.00    555719
   macro avg       0.87      0.76      0.81    555719
weighted avg       1.00      1.00      1.00    555719

Confusion Matrix (Unseen):
[[553189    385]
 [  1009   1136]]
True Positives (TP) (Unseen): 1136
False Positives (FP) (Unseen): 385
True Negatives (TN) (Unseen): 553189
False Negatives (FN) (Unseen): 1009


***************************

In [16]:
%%time
from imblearn.over_sampling import ADASYN
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc, classification_report, confusion_matrix
import numpy as np

# Initialize the tuned base models
tuned_xgb_model = xgb.XGBClassifier(**xgb_bayes_search.best_params_)
tuned_lgb_model = lgb.LGBMClassifier(**lgb_bayes_search.best_params_)

# Define base models
base_models = [
    ('xgb', tuned_xgb_model),
    ('lgb', tuned_lgb_model)
]

# Define stacking classifier
stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())

# Apply ADASYN
adasyn = ADASYN(random_state=42)
X_resampled, y_resampled = adasyn.fit_resample(X_train_scaled_df, y_train)

# Train stacking model with resampled data
stack_model.fit(X_resampled, y_resampled)

# Model Evaluation on Validation Data
y_pred = stack_model.predict(X_val_scaled_df)
y_pred_prob = stack_model.predict_proba(X_val_scaled_df)[:, 1]

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

conf_matrix = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = conf_matrix.ravel()

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_val, y_pred))
print("Confusion Matrix:")
print(conf_matrix)
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'True Negatives (TN): {tn}')
print(f'False Negatives (FN): {fn}')


[LightGBM] [Info] Number of positive: 903167, number of negative: 902418
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007250 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2802
[LightGBM] [Info] Number of data points in the train set: 1805585, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500207 -> initscore=0.000830
[LightGBM] [Info] Start training from score 0.000830
[LightGBM] [Info] Number of positive: 722534, number of negative: 721934
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008124 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2804
[LightGBM] [Info] Number of data points in the train set: 1444468, number of used features: 12
[LightGBM] [

In [17]:
# Model Evaluation on Unseen Data
y_unseen_pred = stack_model.predict(X_unseen_scaled_df)
y_unseen_pred_prob = stack_model.predict_proba(X_unseen_scaled_df)[:, 1]

accuracy_unseen = accuracy_score(y_unseen, y_unseen_pred)
precision_unseen = precision_score(y_unseen, y_unseen_pred)
recall_unseen = recall_score(y_unseen, y_unseen_pred)
f1_unseen = f1_score(y_unseen, y_unseen_pred)
roc_auc_unseen = roc_auc_score(y_unseen, y_unseen_pred_prob)
precision_vals_unseen, recall_vals_unseen, _ = precision_recall_curve(y_unseen, y_unseen_pred_prob)
pr_auc_unseen = auc(recall_vals_unseen, precision_vals_unseen)

conf_matrix_unseen = confusion_matrix(y_unseen, y_unseen_pred)
tn_unseen, fp_unseen, fn_unseen, tp_unseen = conf_matrix_unseen.ravel()

print(f'Accuracy (Unseen): {accuracy_unseen:.4f}')
print(f'Precision (Unseen): {precision_unseen:.4f}')
print(f'Recall (Unseen): {recall_unseen:.4f}')
print(f'F1 Score (Unseen): {f1_unseen:.4f}')
print(f'ROC-AUC (Unseen): {roc_auc_unseen:.4f}')
print(f'PR AUC (Unseen): {pr_auc_unseen:.4f}')
print("Classification Report (Unseen):")
print(classification_report(y_unseen, y_unseen_pred))
print("Confusion Matrix (Unseen):")
print(conf_matrix_unseen)
print(f'True Positives (TP) (Unseen): {tp_unseen}')
print(f'False Positives (FP) (Unseen): {fp_unseen}')
print(f'True Negatives (TN) (Unseen): {tn_unseen}')
print(f'False Negatives (FN) (Unseen): {fn_unseen}')

Accuracy (Unseen): 0.9973
Precision (Unseen): 0.6905
Recall (Unseen): 0.5399
F1 Score (Unseen): 0.6060
ROC-AUC (Unseen): 0.9615
PR AUC (Unseen): 0.4962
Classification Report (Unseen):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.69      0.54      0.61      2145

    accuracy                           1.00    555719
   macro avg       0.84      0.77      0.80    555719
weighted avg       1.00      1.00      1.00    555719

Confusion Matrix (Unseen):
[[553055    519]
 [   987   1158]]
True Positives (TP) (Unseen): 1158
False Positives (FP) (Unseen): 519
True Negatives (TN) (Unseen): 553055
False Negatives (FN) (Unseen): 987


***************

In [19]:
%%time
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc, classification_report, confusion_matrix
from imblearn.over_sampling import ADASYN
import numpy as np

# Initialize the tuned base models
tuned_xgb_model = xgb.XGBClassifier(**xgb_bayes_search.best_params_)
tuned_lgb_model = lgb.LGBMClassifier(**lgb_bayes_search.best_params_)

# Additional models
rf_model = RandomForestClassifier(random_state=42)
gb_model = GradientBoostingClassifier(random_state=42)

# Define base models
base_models = [
    ('xgb', tuned_xgb_model),
    ('lgb', tuned_lgb_model),
    ('rf', rf_model),
    ('gb', gb_model)
]

# Define stacking classifier
stack_model_multi = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())

# Apply ADASYN
adasyn = ADASYN(random_state=42)
X_resampled, y_resampled = adasyn.fit_resample(X_train_scaled_df, y_train)

# Train stacking model with resampled data
stack_model_multi.fit(X_resampled, y_resampled)


[LightGBM] [Info] Number of positive: 903167, number of negative: 902418
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006880 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2802
[LightGBM] [Info] Number of data points in the train set: 1805585, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500207 -> initscore=0.000830
[LightGBM] [Info] Start training from score 0.000830
[LightGBM] [Info] Number of positive: 722534, number of negative: 721934
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007867 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2804
[LightGBM] [Info] Number of data points in the train set: 1444468, number of used features: 12
[LightGBM] [

StackingClassifier(estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_...
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=200, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=None, ...)),
                               ('lgb',
                                LGBMClassifier(learning_rate=0.029123317607560525,
                                               max_depth=30, n_estimators=200,
                                               scale_pos_weight=1)),
                               ('rf', RandomForestClassifier(random_state=42)),
                               ('gb',
                                GradientBoostingClassifier(random_state=42))],
                   final_estimator=LogisticRegression())

In [20]:
%%time
# Model Evaluation on Validation Data
y_pred = stack_model_multi.predict(X_val_scaled_df)
y_pred_prob = stack_model_multi.predict_proba(X_val_scaled_df)[:, 1]

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

conf_matrix = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = conf_matrix.ravel()

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_val, y_pred))
print("Confusion Matrix:")
print(conf_matrix)
print(f'True Positives (TP): {tp}')
print(f'False Positives (FP): {fp}')
print(f'True Negatives (TN): {tn}')
print(f'False Negatives (FN): {fn}')

Accuracy: 0.9977
Precision: 0.7785
Recall: 0.8477
F1 Score: 0.8116
ROC-AUC: 0.9742
PR AUC: 0.8851
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    386751
           1       0.78      0.85      0.81      2252

    accuracy                           1.00    389003
   macro avg       0.89      0.92      0.91    389003
weighted avg       1.00      1.00      1.00    389003

Confusion Matrix:
[[386208    543]
 [   343   1909]]
True Positives (TP): 1909
False Positives (FP): 543
True Negatives (TN): 386208
False Negatives (FN): 343
CPU times: total: 27.5 s
Wall time: 10.8 s


In [21]:
%%time
# Model Evaluation on Unseen Data
y_unseen_pred = stack_model_multi.predict(X_unseen_scaled_df)
y_unseen_pred_prob = stack_model_multi.predict_proba(X_unseen_scaled_df)[:, 1]

accuracy_unseen = accuracy_score(y_unseen, y_unseen_pred)
precision_unseen = precision_score(y_unseen, y_unseen_pred)
recall_unseen = recall_score(y_unseen, y_unseen_pred)
f1_unseen = f1_score(y_unseen, y_unseen_pred)
roc_auc_unseen = roc_auc_score(y_unseen, y_unseen_pred_prob)
precision_vals_unseen, recall_vals_unseen, _ = precision_recall_curve(y_unseen, y_unseen_pred_prob)
pr_auc_unseen = auc(recall_vals_unseen, precision_vals_unseen)

conf_matrix_unseen = confusion_matrix(y_unseen, y_unseen_pred)
tn_unseen, fp_unseen, fn_unseen, tp_unseen = conf_matrix_unseen.ravel()

print(f'Accuracy (Unseen): {accuracy_unseen:.4f}')
print(f'Precision (Unseen): {precision_unseen:.4f}')
print(f'Recall (Unseen): {recall_unseen:.4f}')
print(f'F1 Score (Unseen): {f1_unseen:.4f}')
print(f'ROC-AUC (Unseen): {roc_auc_unseen:.4f}')
print(f'PR AUC (Unseen): {pr_auc_unseen:.4f}')
print("Classification Report (Unseen):")
print(classification_report(y_unseen, y_unseen_pred))
print("Confusion Matrix (Unseen):")
print(conf_matrix_unseen)
print(f'True Positives (TP) (Unseen): {tp_unseen}')
print(f'False Positives (FP) (Unseen): {fp_unseen}')
print(f'True Negatives (TN) (Unseen): {tn_unseen}')
print(f'False Negatives (FN) (Unseen): {fn_unseen}')

Accuracy (Unseen): 0.9966
Precision (Unseen): 0.6374
Recall (Unseen): 0.2811
F1 Score (Unseen): 0.3902
ROC-AUC (Unseen): 0.7627
PR AUC (Unseen): 0.3722
Classification Report (Unseen):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.64      0.28      0.39      2145

    accuracy                           1.00    555719
   macro avg       0.82      0.64      0.69    555719
weighted avg       1.00      1.00      1.00    555719

Confusion Matrix (Unseen):
[[553231    343]
 [  1542    603]]
True Positives (TP) (Unseen): 603
False Positives (FP) (Unseen): 343
True Negatives (TN) (Unseen): 553231
False Negatives (FN) (Unseen): 1542
CPU times: total: 34.2 s
Wall time: 11.5 s


In [22]:
# Check class distribution in unseen data
print("Class distribution in unseen data:")
print(y_unseen.value_counts(normalize=True))

# Compare with class distribution in training data
print("\nClass distribution in training data:")
print(y_train.value_counts(normalize=True))

Class distribution in unseen data:
is_fraud
0    0.99614
1    0.00386
Name: proportion, dtype: float64

Class distribution in training data:
is_fraud
0    0.994212
1    0.005788
Name: proportion, dtype: float64


In [26]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Set the figure size
plt.figure(figsize=(50, 80))  # Adjust the width and height as needed

# Plot the feature importance
plot_importance(xgb_model, importance_type='gain')

# Show the plot
plt.show()


NotFittedError: need to call fit or load_model beforehand

<Figure size 5000x8000 with 0 Axes>

In [25]:
%whos

Variable                     Type                          Data/Info
--------------------------------------------------------------------
ADASYN                       ABCMeta                       <class 'imblearn.over_sampling._adasyn.ADASYN'>
BayesSearchCV                ABCMeta                       <class 'skopt.searchcv.BayesSearchCV'>
GradientBoostingClassifier   ABCMeta                       <class 'sklearn.ensemble.<...>dientBoostingClassifier'>
Integer                      type                          <class 'skopt.space.space.Integer'>
LogisticRegression           type                          <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
RandomForestClassifier       ABCMeta                       <class 'sklearn.ensemble.<...>.RandomForestClassifier'>
Real                         type                          <class 'skopt.space.space.Real'>
SMOTE                        ABCMeta                       <class 'imblearn.over_sam<...>pling._smote.base.SMOTE'>
StackingCl